# Create analysis grid for accessibility analysis
Zehui Yin

In [ ]:
import geopandas as gpd
import h3
from shapely.geometry import Polygon, mapping

In [ ]:
import os
import sys

os.environ['GDAL_DATA'] = os.path.join(f'{os.sep}'.join(sys.executable.split(os.sep)[:-1]), 
                                       'Library', 'share', 'gdal')

## Load city boundaries

In [ ]:
toronto = gpd.read_file("data/Toronto-Boundary/citygcs_regional_mun_wgs84.shp")
hamilton = gpd.read_file("data/Hamilton_City_Boundary.geojson")
hamilton_waterbody = gpd.read_file("data/Waterbodies.geojson")

# Ensure both are in local projection (UTM Zone 17N, EPSG:26917)
toronto = toronto.to_crs(epsg=26917)
hamilton = hamilton.to_crs(epsg=26917)
hamilton_waterbody = hamilton_waterbody.to_crs(epsg=26917)

print(f"Toronto: {toronto.shape[0]} feature(s), CRS: {toronto.crs}")
print(f"Hamilton: {hamilton.shape[0]} feature(s), CRS: {hamilton.crs}")

## Build study area (remove waterbody and union of both cities)

In [ ]:
# Build study area by unioning shapes first, then computing difference (avoid per-feature overlay)
# Compute unary unions (single geometry) and perform a shapely difference on the unioned geometry
hamilton_union = hamilton.geometry.union_all()
hamilton_water_union = hamilton_waterbody.geometry.union_all()
hamilton_non_water_geom = hamilton_union.difference(hamilton_water_union)

# Combine with Toronto union
origin_geometry = toronto.geometry.union_all().union(hamilton_non_water_geom)
study_area = origin_geometry.buffer(1000)  # Add 1km buffer to ensure full coverage

study_area_gdf = gpd.GeoDataFrame(
    {'name': ['Toronto + Hamilton']},
    geometry=[study_area],
    crs='EPSG:26917'
).to_crs(epsg=4326)  # Convert to WGS84 for H3 processing

print(f"Study area geometry type: {study_area_gdf.geom_type.iloc[0]}")
print(f"Bounding box: {study_area_gdf.bounds.iloc[0]}")

## Generate H3 hexagonal grid (resolution 9)

Resolution 9 provides fine-scale hexagons with an average area of ~0.105 km² and edge length of ~200 m, suitable for urban transit accessibility analysis.

In [ ]:
H3_RESOLUTION = 9

print(f"H3 resolution {H3_RESOLUTION} statistics:")
print(f"  Average hexagon area : {h3.average_hexagon_area(H3_RESOLUTION, unit='km^2'):.4f} km²")
print(f"  Average edge length  : {h3.average_hexagon_edge_length(H3_RESOLUTION, unit='km'):.4f} km")

In [ ]:
# Convert study area polygon to H3 shape and fill with hexagons
h3shape = h3.geo_to_h3shape(mapping(study_area_gdf.iloc[0].geometry))
cells = h3.h3shape_to_cells(h3shape, res=H3_RESOLUTION)

print(f"Total H3 cells generated: {len(cells):,}")

## Convert H3 cells to GeoDataFrame

In [ ]:
def cell_to_polygon(cell):
    """Convert an H3 cell index to a Shapely Polygon."""
    coords = h3.cell_to_boundary(cell)  # returns (lat, lng) tuples
    return Polygon([(lng, lat) for lat, lng in coords])

grid_gdf = gpd.GeoDataFrame(
    {'h3_id': list(cells), 'h3_res': H3_RESOLUTION},
    geometry=[cell_to_polygon(c) for c in cells],
    crs='EPSG:4326'
)

print(grid_gdf.shape)
grid_gdf.head()

In [ ]:
# Filter grid to only those that intersect the original study area
grid_gdf = grid_gdf[grid_gdf.to_crs(epsg=26917).intersects(origin_geometry)].reset_index(drop=True)
print(f"Grid cells after intersection filter: {grid_gdf.shape[0]:,}")

## Visualise the grid over the study area

In [ ]:
grid_gdf.explore()

## Save grid to file

In [ ]:
output_path = "output/analysis_grid_h3_res9.geojson"
grid_gdf.to_file(output_path, driver="GeoJSON")
print(f"Grid saved to: {output_path}")
print(f"Features: {len(grid_gdf):,}")
print(f"CRS: {grid_gdf.crs}")